# SmolVLM: test-set mean visual attention (base vs reverse)
FP32, two GPUs, shared read-only Kaggle feature cache. Mean heads then mean all 16,817 test samples. The final prompt position predicts the first answer token. No training, raw-image extraction, generation or dataset publishing.

Select **GPU T4 ×2** (or another two-GPU accelerator), enable Internet, and attach `huyhuy12334/cache-smolvlm` as Input. This notebook bundles its Python source: no GitHub push/checkout is required. Keep the two launch cells separate; they return immediately, so both models run concurrently. Each plot type and model has its own cell. The last cell exports a ZIP.

In [ ]:
# 1. Bootstrap bundled source and dependencies; keep Kaggle CUDA Torch.
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/kaggle/working/.cache/huggingface'
os.environ['PIP_CACHE_DIR'] = '/kaggle/working/.cache/pip'
from pathlib import Path
import base64, hashlib, io, json, subprocess, sys, zipfile
SOURCE = Path('/kaggle/working/smolvlm-attention-source')
SOURCE_COMMIT = 'a40c205db4ee28b088cd6e2d528a858361c6d274'
SOURCE_ARCHIVE_SHA256 = '7a2802c6b4e98c6ed80c446cbe1e404f554fd3b250f2c5ccec4d07d9bf39e3b6'
SOURCE_B64 = '...'
source_bytes = base64.b64decode(SOURCE_B64)
assert hashlib.sha256(source_bytes).hexdigest() == SOURCE_ARCHIVE_SHA256
SOURCE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
    for name in archive.namelist():
        target = (SOURCE / name).resolve()
        if not target.is_relative_to(SOURCE.resolve()):
            raise ValueError('Unsafe bundled source path')
    archive.extractall(SOURCE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers',
                '-r', str(SOURCE / 'smolvlm_attention/requirements.txt')], check=True)
sys.path.insert(0, str(SOURCE))
import torch, transformers, tokenizers
if transformers.__version__ != '4.46.3' or tokenizers.__version__ != '0.20.4':
    raise RuntimeError('Restart kernel after installing pinned packages, then Run All')
if torch.cuda.device_count() < 2:
    raise RuntimeError('Enable an accelerator with two GPUs for this notebook')
print({'torch': torch.__version__, 'source_sha256': SOURCE_ARCHIVE_SHA256,
       'gpus': [torch.cuda.get_device_name(i) for i in range(2)], 'precision': 'float32'})

In [ ]:
# 2. Pinned artifacts. Existing mounted cache is never copied into working.
from smolvlm_attention.runtime import resolve_artifacts
OUTPUT = Path('/kaggle/working/attention_outputs')
CACHE_INPUT_DIR = ''  # Optional: mounted directory containing the cache files.
artifacts = resolve_artifacts(OUTPUT, CACHE_INPUT_DIR)
print({name: spec for name, spec in artifacts['models'].items()})
print({'cache': artifacts['cache_dir'], 'prepared': artifacts['prepared_dir']})

In [ ]:
# 3. Full cache validation ONCE, then pretokenize test questions in both prompt orders.
from smolvlm_attention.runtime import prepare_run, launch_worker, wait_workers, load_results
RUN_CONFIG = prepare_run(artifacts, source_commit=SOURCE_COMMIT,
                         source_archive_sha256=SOURCE_ARCHIVE_SHA256)
print(RUN_CONFIG)

In [ ]:
# 4. Base: launch asynchronously on physical GPU 0.
base_worker = launch_worker(RUN_CONFIG, "base")

In [ ]:
# 5. Reverse: launch asynchronously on physical GPU 1.
reverse_worker = launch_worker(RUN_CONFIG, "reverse")

In [ ]:
# 6. Wait for both workers. Logs contain progress, selected batch, parity checks and errors.
# Interrupting this wait cell leaves workers running; rerun it to reconnect.
wait_workers(RUN_CONFIG)

In [ ]:
# 7. Require full test coverage for both models and compute SHARED plot scales.
means, model_metadata, scales, figure_metadata = load_results(RUN_CONFIG)
FIGURES = OUTPUT / 'figures'
from smolvlm_attention.plotting import (plot_token_atlas, plot_spatial_atlas,
                                       plot_layer_histogram, make_figures_zip)
from IPython.display import display, Image, FileLink
orders = {name: spec['camera_order'] for name, spec in artifacts['models'].items()}
print({name: {'count': model_metadata[name]['count'], 'shape': list(value.shape)}
       for name, value in means.items()})

In [ ]:
# BASE: 384 tokens, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_token_atlas(means["base"], orders["base"], FIGURES / "base",
    title='base: 384 tokens, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# BASE: six camera maps per layer, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_spatial_atlas(means["base"], orders["base"], FIGURES / "base",
    title='base: six camera maps per layer, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# BASE: distribution of 384 sample-averaged token values, all 30 layers. Re-run plotting without rerunning measurement.
paths = plot_layer_histogram(means["base"], orders["base"], FIGURES / "base",
    title='base: distribution of 384 sample-averaged token values, all 30 layers', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# REVERSE: 384 tokens, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_token_atlas(means["reverse"], orders["reverse"], FIGURES / "reverse",
    title='reverse: 384 tokens, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# REVERSE: six camera maps per layer, layers 0,2,...,28. Re-run plotting without rerunning measurement.
paths = plot_spatial_atlas(means["reverse"], orders["reverse"], FIGURES / "reverse",
    title='reverse: six camera maps per layer, layers 0,2,...,28', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# REVERSE: distribution of 384 sample-averaged token values, all 30 layers. Re-run plotting without rerunning measurement.
paths = plot_layer_histogram(means["reverse"], orders["reverse"], FIGURES / "reverse",
    title='reverse: distribution of 384 sample-averaged token values, all 30 layers', scales=scales)
display(Image(filename=str(paths[0])))
print([str(path) for path in paths])

In [ ]:
# 14. ZIP contains both figure folders (PNG/PDF) and measurement metadata only.
zip_path = make_figures_zip(FIGURES, OUTPUT / 'attention_figures.zip', figure_metadata)
print(f'{zip_path} | {zip_path.stat().st_size / 1024**2:.2f} MiB')
try:
    download_path = zip_path.relative_to(Path.cwd())
except ValueError:
    download_path = zip_path
display(FileLink(str(download_path)))

## Interpretation
The base best checkpoint is epoch 2; reverse is epoch 1. Compare their sample-averaged attention patterns, not a controlled camera-order intervention on identical weights. Histogram colors count averaged visual-token positions (384 per layer), not samples or heads. Camera heatmaps describe mean spatial positions, not any one source image.

If CUDA OOM occurs, workers reduce batch size without dropping records or switching precision. Progress snapshots allow matching runs to resume. A failed native parity check stops measurement; read `attention_outputs/logs/`. Outputs are complete only when both models cover every test ID.